# Phase 3 — GPU run

Difficulty-Aware Adaptive Reasoning for Financial Advisory Systems.

**Sidebar settings, before you start:**

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 x2** or **GPU P100** |
| Internet | **On** (needs phone verification on your Kaggle account) |
| Input | the `adaptive-reasoning-fas` dataset attached |

## Run it with "Save Version → Save & Run All"

Do **not** babysit this interactively. The run takes ~4.5 hours, and an interactive
kernel restart wipes `/kaggle/working` — losing every checkpoint with it. That happened
once already.

**Save Version → Save & Run All (Commit)** runs the whole notebook in a background
session of up to 12 hours and preserves the output regardless of your browser.

Cell 4 is a gate: it verifies the model actually reasons at length before the long run
starts. If it fails, the run stops there rather than producing hours of unusable data.

## 1. Unpack the project

In [ ]:
import glob
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

WORK = Path('/kaggle/working')
PROJECT = WORK / 'project'
INPUT = Path('/kaggle/input')

# Kaggle sometimes stores an uploaded archive as-is and sometimes auto-extracts it,
# so both layouts are handled. The project root is whatever directory contains
# configs/default.yaml.
if PROJECT.exists():
    shutil.rmtree(PROJECT)

zips = glob.glob(str(INPUT / '**' / 'adaptive-reasoning.zip'), recursive=True)
markers = glob.glob(str(INPUT / '**' / 'configs' / 'default.yaml'), recursive=True)

if zips:
    print(f'found archive: {zips[0]}')
    with zipfile.ZipFile(zips[0]) as zf:
        zf.extractall(WORK)
    # The archive stores everything under project/.
    if not PROJECT.exists():
        found = glob.glob(str(WORK / '**' / 'configs' / 'default.yaml'), recursive=True)
        assert found, 'archive extracted but configs/default.yaml is missing'
        Path(found[0]).parent.parent.rename(PROJECT)

elif markers:
    source = Path(markers[0]).parent.parent
    print(f'found extracted project: {source}')
    # /kaggle/input is read-only, so copy into the writable working directory.
    shutil.copytree(source, PROJECT)

else:
    listing = sorted(str(p) for p in INPUT.glob('*'))
    detail = '\n'.join(f'    {p}' for p in listing) or '    (nothing attached)'
    raise SystemExit(
        'Could not find the project under /kaggle/input.\n\n'
        f'What is attached:\n{detail}\n\n'
        'Fix: in the right-hand sidebar click  + Add Input  ->  Datasets  ->  and\n'
        'attach your "adaptive-reasoning-fas" dataset, then re-run this cell.'
    )

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / 'src'))

# Fail loudly now rather than three cells later.
for required in ['configs/default.yaml', 'src/adaptive_reasoning', 'scripts/run_pilot.py',
                 'data/processed/unified.parquet']:
    assert (PROJECT / required).exists(), f'missing from the upload: {required}'

print('project root:', PROJECT)
print('contents:', sorted(p.name for p in PROJECT.iterdir()))

## 2. Dependencies

Kaggle already ships torch with CUDA — we deliberately do **not** touch it. Only the
few packages the project needs on top are installed.

In [ ]:
%pip install -q --no-warn-conflicts \
    'pydantic>=2.7' 'transformers>=4.44' 'sentence-transformers>=3.0' \
    lightgbm joblib pyarrow rich nvidia-ml-py
print('done')

## 3. Environment check

In [ ]:
import torch

print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}, {p.total_memory/1024**3:.1f} GB')
else:
    raise SystemExit('No GPU. Set Accelerator to GPU in the sidebar and restart.')

!python scripts/check_env.py

## 4. PILOT GATE — do not skip

Two minutes. Checks the two assumptions Phase 3 rests on:

1. **The model reasons at length.** If it answers in a handful of tokens there is
   nothing to stop early and no signal for the DQN. A plain instruction model fails
   here — measured locally, `Qwen2.5-0.5B-Instruct` had a *median of 3 reasoning
   tokens*.
2. **Throughput is what we assumed.** Every GPU-hour estimate in the docs is a guess
   until this measures it. The pilot prints a projected cost from observed tokens/sec.

A non-zero exit means **stop here**.

In [ ]:
rc = subprocess.call([sys.executable, 'scripts/run_pilot.py'])
print('\nexit code:', rc)
if rc != 0:
    print('\nGATE FAILED — do not run the cells below.')
    print('Try a reasoning-tuned model, e.g.:')
    print("  !python scripts/run_pilot.py --model deepseek-ai/DeepSeek-R1-Distill-Qwen-7B")

## 5. Tuned settings — run 2

The first run capped reasoning at 768 tokens to fit the free GPU quota. Measured
afterwards, that cap was the single largest cause of low accuracy:

| Questions | Share | Accuracy |
|---|---|---|
| Reasoning completed | 75% | **53.5%** |
| Cut off at the cap | 25% | **9.5%** |

Worst affected were the synthetic finance-maths questions (68% truncated) and credit
risk (39% truncated) — both of which have reachable answers the model was simply not
given room to finish.

**This run doubles the budget to 1536 tokens.** Expected effect: overall accuracy from
42% to roughly 52%.

* `max_new_tokens: 1536` — the change that matters
* `max_steps: 32` — twice as many probe points, so the longer traces are still covered
  at the same 48-token resolution
* `n_questions: 4000` and seed unchanged, so this run traces the **same 4,000
  questions** as run 1 and the results are directly comparable
* Wall-clock roughly doubles, to about 9 hours — still inside a 12-hour session

Difficulty sampling remains switched off; labels are derived from these traces on CPU
afterwards.

In [ ]:
Path('configs/experiment').mkdir(parents=True, exist_ok=True)
Path('configs/experiment/tuned.yaml').write_text("""llm:
  batch_size: 32
  max_new_tokens: 1536

traces:
  n_questions: 4000
  step_tokens: 48
  max_steps: 32
  probe_max_tokens: 16
  checkpoint_every: 200
""")
print(Path('configs/experiment/tuned.yaml').read_text())

> **Difficulty labelling is not run here.** It is derived from the traces on CPU after
> download, with `python scripts/run_phase2.py --stage label`. Nothing on this GPU
> session needs to produce it.

## 6. Phase 3 — reasoning traces

The main run. Generates full reasoning for each question and probes at every step
boundary to record what the answer *would* have been had it stopped there.

This is the long cell. It also checkpoints, so re-running after a timeout continues
from where it stopped.

In [ ]:
import os

# expandable_segments reduces allocator fragmentation. The first run showed 9.4 GB
# reserved-but-unallocated at the point it ran out of memory.
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python scripts/run_phase3.py --experiment tuned

## 7. Package the results

Everything Phases 4–9 need, zipped into `/kaggle/working` for download. Unzip it over
your local project root and continue on CPU.

In [ ]:
import json

OUT = WORK / 'phase3_results.zip'
WANTED = [
    'artifacts/traces/traces.parquet',
    'artifacts/traces/trace_summary.parquet',
    'artifacts/models/difficulty_clf.joblib',
    'artifacts/results/phase3_pilot.json',
    'artifacts/results/phase3_summary.json',
    'artifacts/results/phase2_difficulty_classifier.json',
    'artifacts/results/difficulty_sampling_manifest.json',
    'data/processed/unified.parquet',
    'data/processed/difficulty_labels.parquet',
    'data/processed/difficulty_samples.parquet',
]

with zipfile.ZipFile(OUT, 'w', zipfile.ZIP_DEFLATED) as zf:
    for rel in WANTED:
        path = PROJECT / rel
        if path.exists():
            zf.write(path, rel)
            print(f'  + {rel}  ({path.stat().st_size/1e6:.1f} MB)')
        else:
            print(f'  - {rel}  MISSING')

print(f'\nwrote {OUT}  ({OUT.stat().st_size/1e6:.1f} MB)')
print('Download it from the Output panel on the right.')

summary = PROJECT / 'artifacts/results/phase3_summary.json'
if summary.exists():
    print('\n--- Phase 3 summary ---')
    print(json.dumps(json.loads(summary.read_text()), indent=2))